# LeetCode 250 — Count Univalue Subtrees bằng Python

Link đề: https://leetcode.com/problems/count-univalue-subtrees/ · Độ khó: **Medium** (Premium) · Chủ đề: Binary Tree, DFS (hậu thứ tự), Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def countUnivalSubtrees(self, root: Optional[TreeNode]) -> int:
        ...
```

Ghi chú: đề bài viết theo hiểu biết về bài (bài Premium), nên đối chiếu lại nếu có tài khoản Premium; ví dụ và constraints lấy từ file đề có sẵn. Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (gom mọi giá trị của từng cây con).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số **cây con "đơn trị" (uni-value subtree)** — cây con mà **mọi nút có cùng một giá trị**; mỗi **nút** của cây gốc xác định đúng một cây con (gồm nó và toàn bộ con cháu).
2. Hiểu **cây con** khác **cây con bất kỳ tập nút**: cây con ở nút `v` luôn lấy **cả** con cháu của `v`.
3. Đi từ cách **với mỗi nút, duyệt cả cây con kiểm tra** `O(n²)` → **DFS hậu thứ tự trả về "cây con này có đơn trị không"** `O(n)` → **trả về cả (số lượng, đơn trị)** không dùng biến ngoài → **hậu thứ tự lặp bằng stack**.
4. Tự dựng `TreeNode`, `build_tree` (danh sách level-order kiểu LeetCode) và nắm tư duy **"con trả lời trước, cha tổng hợp sau"** (post-order).
5. Tránh các lỗi kinh điển: chỉ so giá trị cha với giá trị con mà **không** yêu cầu con đơn trị, **ngắt sớm** (`or`) làm bỏ sót đếm nhánh phải, so sánh với `None`, đếm nhầm cây rỗng.

## 2. Đề bài & ví dụ

Cho `root` của một cây nhị phân, trả về **số lượng cây con đơn trị** (uni-value subtree).

Một **cây con đơn trị** là cây con mà **mọi nút của nó có cùng giá trị**.

**Ví dụ 1**

- Input: `root = [5,1,5,5,5,null,5]`
- Output: `4`
- Giải thích: các cây con đơn trị là ba lá `5` (hai lá dưới nút `1` và lá `5` ở cuối nhánh phải) và cây con gốc tại nút `5` bên phải (gồm nút `5` đó và con phải `5`); tổng `4`.

**Ví dụ 2**

- Input: `root = []`
- Output: `0`

**Ví dụ 3**

- Input: `root = [5,5,5,5,5,null,5]`
- Output: `6`
- Giải thích: mọi cây con đều đơn trị (tất cả nút đều là `5`), cây có `6` nút nên có `6` cây con.

**Ràng buộc:** số nút trong cây thuộc `[0, 1000]`; `-1000 <= Node.val <= 1000`.

### Dựng cây trong notebook

LeetCode cho sẵn `TreeNode` và biểu diễn cây theo **level-order**, `null` là nút rỗng. Ta tự dựng lại `TreeNode`, `build_tree`, `to_list`. Ví dụ `[5,1,5,5,5,null,5]`:

```
        5
      /   \
     1     5
    / \     \
   5   5     5
```

In [1]:
from typing import Optional, List, Tuple
from collections import deque
import random


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values: List[Optional[int]]) -> Optional[TreeNode]:
    """Dựng cây từ danh sách level-order kiểu LeetCode (None = nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def to_list(root: Optional[TreeNode]) -> List[Optional[int]]:
    """Chuyển cây về danh sách level-order (bỏ các None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([5, 1, 5, 5, 5, None, 5], 4),
    ([], 0),
    ([5, 5, 5, 5, 5, None, 5], 6),
]
for values, expected in examples:
    print(f"root={values} -> dựng lại {to_list(build_tree(values))} -> kỳ vọng {expected}")

root=[5, 1, 5, 5, 5, None, 5] -> dựng lại [5, 1, 5, 5, 5, None, 5] -> kỳ vọng 4
root=[] -> dựng lại [] -> kỳ vọng 0
root=[5, 5, 5, 5, 5, None, 5] -> dựng lại [5, 5, 5, 5, 5, None, 5] -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1. Cây con (subtree) của một nút

Cây con tại nút `v` gồm `v` **và mọi con cháu** của `v`. Cây có `n` nút thì có đúng `n` cây con (một cho mỗi nút). Lá là cây con một nút — **luôn đơn trị** (chỉ có một giá trị). Cây rỗng (`None`) **không** được tính là một cây con của đề.
#### Code demo

In [2]:
root = build_tree([5, 1, 5, 5, 5, None, 5])


def subtree_values(node):
    """Danh sách giá trị của cả cây con tại node (duyệt tiền thứ tự)."""
    if node is None:
        return []
    return [node.val] + subtree_values(node.left) + subtree_values(node.right)


print("cây con tại gốc     :", subtree_values(root))             # [5, 1, 5, 5, 5, 5]
print("cây con tại nút 1   :", subtree_values(root.left))        # [1, 5, 5]
print("cây con tại nút 5 phải:", subtree_values(root.right))     # [5, 5]
print("cây con tại lá      :", subtree_values(root.left.left))   # [5]

cây con tại gốc     : [5, 1, 5, 5, 5, 5]
cây con tại nút 1   : [1, 5, 5]
cây con tại nút 5 phải: [5, 5]
cây con tại lá      : [5]


### 3.2. Đơn trị: `set` của giá trị chỉ có một phần tử

Một cây con đơn trị khi `len(set(giá_trị)) == 1`. Cách này đơn giản nhưng tốn công (gom hết giá trị). Ta sẽ tìm cách nhanh hơn bằng việc **tái sử dụng kết quả của cây con con**.
#### Code demo

In [3]:
for name, node in [("gốc", root), ("nút 1", root.left), ("nút 5 phải", root.right), ("lá", root.left.left)]:
    vals = subtree_values(node)
    print(f"{name:>10}: {vals} -> {len(set(vals)) == 1}")

       gốc: [5, 1, 5, 5, 5, 5] -> False
     nút 1: [1, 5, 5] -> False
nút 5 phải: [5, 5] -> True
        lá: [5] -> True


### 3.3. Đệ quy hậu thứ tự (post-order)

Duyệt **hậu thứ tự**: xử lý con trái, con phải **trước**, rồi mới xử lý nút hiện tại. Phù hợp khi kết quả của cha **phụ thuộc** vào kết quả của hai con. Mẫu: `left_result = f(node.left)`, `right_result = f(node.right)`, rồi kết hợp. Cơ sở: `f(None)`.
#### Code demo

In [4]:
def post_order(node, out):
    if node is None:
        return
    post_order(node.left, out)
    post_order(node.right, out)
    out.append(node.val)                 # xử lý nút SAU hai con


out = []
post_order(build_tree([1, 2, 3, 4, 5]), out)
print(out)                               # [4, 5, 2, 3, 1]

[4, 5, 2, 3, 1]


### 3.4. Điều kiện đơn trị theo "con trả lời trước"

Nút `v` có cây con đơn trị ⟺ **cả hai** điều sau: (a) mỗi con **có** (nếu tồn tại) là cây con đơn trị, và (b) mỗi con tồn tại có **giá trị bằng `v.val`**. Chỉ (b) là chưa đủ: `5` có con `5` mà con đó lại có con `1` thì không đơn trị. Con vắng mặt (`None`) coi như **không vi phạm** (đơn trị "rỗng").
#### Code demo

In [5]:
leaf_ok = build_tree([5, 5, 1])          # gốc 5, con trái 5, con phải 1
print(root.val, "so với con:", root.left.val, root.right.val)

tricky = build_tree([5, 5, None, 1])     # gốc 5 -> con trái 5 -> con trái của nó là 1
print("tricky:", to_list(tricky), "| giá trị con trái của gốc =", tricky.left.val, "(bằng gốc!) nhưng cây con của nó có", tricky.left.left.val)

5 so với con: 1 5
tricky: [5, 5, None, 1] | giá trị con trái của gốc = 5 (bằng gốc!) nhưng cây con của nó có 1


### 3.5. Biến toàn cục / `nonlocal` và trả về nhiều giá trị

Đếm trong đệ quy có hai cách: (1) dùng biến `count` bên ngoài và `nonlocal count` trong hàm lồng; (2) hàm **trả về tuple** `(số_lượng, có_đơn_trị)` rồi cộng dồn — **không cần biến ngoài**, dễ kiểm thử. Ta sẽ làm cả hai.
#### Code demo

In [6]:
def demo_counter():
    count = 0

    def visit():
        nonlocal count                   # phải khai báo nonlocal để gán lại biến ngoài
        count += 1

    visit()
    visit()
    return count


def demo_tuple():
    return 2, True                       # trả về hai giá trị cùng lúc


print(demo_counter(), demo_tuple())
n, ok = demo_tuple()
print(n, ok)

2 (2, True)
2 True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1, cây `[5,1,5,5,5,null,5]`. Duyệt từ lá lên:

- Ba lá `5` (hai lá dưới nút `1`, một lá dưới nút `5` bên phải): mỗi lá đơn trị ⇒ `3`.
- Nút `1` (con `5`, `5`): giá trị `1` khác con ⇒ **không** đơn trị.
- Nút `5` bên phải (chỉ có con phải `5`, đơn trị, giá trị bằng `5`) ⇒ **đơn trị** ⇒ `+1` (tổng `4`).
- Gốc `5`: con trái (nút `1`) **không** đơn trị ⇒ không đơn trị.

Tổng `4`. Cách nghĩ: **hỏi từng nút "cây con của bạn có đơn trị không?"**, và nút chỉ trả lời "có" nếu **cả hai con đều đơn trị và cùng giá trị với nó**. Thông tin đi **từ dưới lên**.

## 5. Cách 1 — Brute Force / baseline: với mỗi nút, duyệt cả cây con để kiểm tra

**Ý tưởng.** Với mỗi nút `v`, gọi `all_equal(v, v.val)` duyệt **toàn bộ** cây con của `v` xem mọi giá trị có bằng `v.val` không; nếu có thì `+1`. Rồi lặp lại cho các nút con.
#### Code demo

In [7]:
def is_uni(node, val):
    """Mọi nút trong cây con tại node đều có giá trị val?"""
    if node is None:
        return True
    return node.val == val and is_uni(node.left, val) and is_uni(node.right, val)


def count_bruteforce(root):
    if root is None:
        return 0
    here = 1 if is_uni(root, root.val) else 0
    return here + count_bruteforce(root.left) + count_bruteforce(root.right)


for values, expected in examples:
    print(values, "->", count_bruteforce(build_tree(values)), "(kỳ vọng", expected, ")")

[5, 1, 5, 5, 5, None, 5] -> 4 (kỳ vọng 4 )
[] -> 0 (kỳ vọng 0 )
[5, 5, 5, 5, 5, None, 5] -> 6 (kỳ vọng 6 )


**Độ phức tạp (Cách 1).** Gọi `n` là số nút, `h` là chiều cao. Mỗi nút duyệt cây con của nó; tổng bằng tổng kích thước mọi cây con = `O(n·h)` — worst case (cây lệch, `h = n`) là **`O(n²)`**. **Space `O(h)`** (ngăn xếp).

**Nút thắt.** Khi kiểm tra nút `v`, ta **duyệt lại** cây con mà con của `v` vừa kiểm tra xong. Kết quả của con có thể dùng lại.

## 6. Ý tưởng thuật toán / bước đột phá

**Dùng kết quả của con:** nút `v` đơn trị ⟺ (con trái rỗng hoặc đơn trị với giá trị `= v.val`) **và** (con phải rỗng hoặc đơn trị với giá trị `= v.val`). Nếu con đã báo "đơn trị" thì giá trị của **cả cây con** là `child.val` (vì mọi nút bằng nhau), nên chỉ cần so `child.val` với `v.val`.

Vậy hàm `dfs(node)` trả về **bool: cây con tại `node` có đơn trị không** (và cộng `count` khi trả `True`). Duyệt hậu thứ tự: mỗi nút làm `O(1)` việc ⇒ **`O(n)`**.

**Cảnh báo quan trọng:** phải **gọi `dfs` cho cả hai con trước khi quyết định** (để đếm đủ các cây con ở **cả hai nhánh**), không được ngắt sớm khi nhánh trái đã sai.

## 7. Các cách tiếp theo

### Cách 2 — DFS hậu thứ tự trả về bool, đếm bằng biến ngoài

**Giải thích.** `dfs(node)` trả `True` nếu cây con đơn trị. Với `None` trả `True` (rỗng không vi phạm, **không đếm**). Gọi cả `l = dfs(left)`, `r = dfs(right)` **trước**; nếu một trong hai `False` thì trả `False`. Ngược lại kiểm tra giá trị của từng con (nếu tồn tại) bằng `node.val`; nếu khớp thì `count += 1` và trả `True`.
#### Code demo

In [8]:
def count_dfs_bool(root):
    count = 0

    def dfs(node):
        nonlocal count
        if node is None:
            return True
        l = dfs(node.left)                    # gọi CẢ HAI trước khi quyết định (để đếm hết hai nhánh)
        r = dfs(node.right)
        if not l or not r:
            return False
        if node.left is not None and node.left.val != node.val:
            return False
        if node.right is not None and node.right.val != node.val:
            return False
        count += 1
        return True

    dfs(root)
    return count


for values, expected in examples:
    print(values, "->", count_dfs_bool(build_tree(values)), "(kỳ vọng", expected, ")")

[5, 1, 5, 5, 5, None, 5] -> 4 (kỳ vọng 4 )
[] -> 0 (kỳ vọng 0 )
[5, 5, 5, 5, 5, None, 5] -> 6 (kỳ vọng 6 )


In [9]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def countUnivalSubtrees(self, root: Optional[TreeNode]) -> int:
        ans = 0

        def dfs(node: Optional[TreeNode]) -> bool:
            nonlocal ans
            if node is None:
                return True
            l, r = dfs(node.left), dfs(node.right)      # đủ cả hai vế, không dùng 'or' ngắt sớm
            if not l or not r:
                return False
            if node.left and node.left.val != node.val:
                return False
            if node.right and node.right.val != node.val:
                return False
            ans += 1
            return True

        dfs(root)
        return ans


print(Solution().countUnivalSubtrees(build_tree([5, 1, 5, 5, 5, None, 5])))   # 4

4


**Độ phức tạp Cách 2:** **Time `O(n)`** (mỗi nút thăm một lần, việc `O(1)`), **Space `O(h)`** cho ngăn xếp đệ quy (`h` là chiều cao; tối đa `n = 1000` ở cây lệch — chạm giới hạn đệ quy mặc định `1000` nếu nhỏ hơn, xem Cách 4).

### Cách 3 — Hàm thuần túy: trả về `(số_lượng, đơn_trị)` (không biến ngoài)

**Giải thích.** `f(node)` trả `(cnt, uni)`: `cnt` là số cây con đơn trị **trong cây con của `node`**, `uni` cho biết cây con của `node` có đơn trị không. Kết hợp: `cnt = cntL + cntR + (1 nếu node đơn trị)`.
#### Code demo

In [10]:
def count_pure(root):
    def f(node) -> Tuple[int, bool]:
        if node is None:
            return 0, True
        cl, ul = f(node.left)
        cr, ur = f(node.right)
        uni = (ul and ur
               and (node.left is None or node.left.val == node.val)
               and (node.right is None or node.right.val == node.val))
        return cl + cr + (1 if uni else 0), uni

    return f(root)[0]


for values, expected in examples:
    print(values, "->", count_pure(build_tree(values)), "(kỳ vọng", expected, ")")

[5, 1, 5, 5, 5, None, 5] -> 4 (kỳ vọng 4 )
[] -> 0 (kỳ vọng 0 )
[5, 5, 5, 5, 5, None, 5] -> 6 (kỳ vọng 6 )


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(h)`**. Ưu điểm: không phụ thuộc biến ngoài, dễ kiểm thử từng hàm.

### Cách 4 — Hậu thứ tự lặp (không đệ quy) bằng stack

**Giải thích.** Để tránh tràn ngăn xếp đệ quy ở cây sâu, ta tạo **thứ tự hậu thứ tự** bằng stack (tiền thứ tự đảo ngược "gốc, phải, trái" rồi đảo lại), rồi xử lý từ **cuối về đầu**: khi xử lý một nút, hai con đã được xử lý xong và kết quả nằm trong từ điển `uni[node]`.
#### Code demo

In [11]:
def count_iterative(root):
    if root is None:
        return 0
    order, stack = [], [root]
    while stack:                                  # tiền thứ tự: gốc, rồi đẩy trái, phải
        node = stack.pop()
        order.append(node)
        if node.left:
            stack.append(node.left)
        if node.right:
            stack.append(node.right)
    uni, count = {}, 0
    for node in reversed(order):                  # đảo lại: con luôn đứng trước cha
        ok = True
        for child in (node.left, node.right):
            if child is not None and not (uni[child] and child.val == node.val):
                ok = False
        uni[node] = ok
        if ok:
            count += 1
    return count


for values, expected in examples:
    print(values, "->", count_iterative(build_tree(values)), "(kỳ vọng", expected, ")")

[5, 1, 5, 5, 5, None, 5] -> 4 (kỳ vọng 4 )
[] -> 0 (kỳ vọng 0 )
[5, 5, 5, 5, 5, None, 5] -> 6 (kỳ vọng 6 )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(n)`** (danh sách `order` và từ điển `uni`). Đổi lại không phụ thuộc giới hạn đệ quy.

> Cách 2 là bản nộp phổ biến; Cách 3 đẹp về mặt hàm thuần; Cách 4 an toàn cho cây rất sâu.

## 8. TRACE từng bước

Ta in từng nút theo thứ tự hậu thứ tự: giá trị, kết quả hai con, quyết định, và `count`. Cây ví dụ 1 `[5,1,5,5,5,null,5]`.
#### Code demo

In [12]:
def trace_dfs(root):
    count = 0
    nodes_label = {}

    def label(node):
        return f"{node.val}" if node is None else f"{node.val}@{id(node) % 1000}"

    def dfs(node, depth=0):
        nonlocal count
        pad = "  " * depth
        if node is None:
            return True
        l = dfs(node.left, depth + 1)
        r = dfs(node.right, depth + 1)
        reasons = []
        if not l:
            reasons.append("con trái không đơn trị")
        if not r:
            reasons.append("con phải không đơn trị")
        if node.left is not None and node.left.val != node.val:
            reasons.append(f"con trái {node.left.val} != {node.val}")
        if node.right is not None and node.right.val != node.val:
            reasons.append(f"con phải {node.right.val} != {node.val}")
        ok = not reasons
        if ok:
            count += 1
        print(f"{pad}nút {node.val}: trái={'-' if node.left is None else l}, phải={'-' if node.right is None else r} -> "
              f"{'ĐƠN TRỊ (count=' + str(count) + ')' if ok else 'không (' + '; '.join(reasons) + ')'}")
        return ok

    dfs(root)
    print("tổng:", count)
    return count


trace_dfs(build_tree([5, 1, 5, 5, 5, None, 5]))

    nút 5: trái=-, phải=- -> ĐƠN TRỊ (count=1)
    nút 5: trái=-, phải=- -> ĐƠN TRỊ (count=2)
  nút 1: trái=True, phải=True -> không (con trái 5 != 1; con phải 5 != 1)
    nút 5: trái=-, phải=- -> ĐƠN TRỊ (count=3)
  nút 5: trái=-, phải=True -> ĐƠN TRỊ (count=4)
nút 5: trái=False, phải=True -> không (con trái không đơn trị; con trái 1 != 5)
tổng: 4


4

Bảng tóm tắt (ví dụ 1; xử lý theo hậu thứ tự):

| nút | con trái | con phải | kết luận | `count` |
|---|---|---|---|---|
| lá `5` (trái của `1`) | — | — | đơn trị | 1 |
| lá `5` (phải của `1`) | — | — | đơn trị | 2 |
| nút `1` | `5` | `5` | `5 != 1`: không | 2 |
| lá `5` (phải của nút `5` phải) | — | — | đơn trị | 3 |
| nút `5` phải | — | `5` (đơn trị) | đơn trị | 4 |
| gốc `5` | nút `1` (không) | nút `5` | con trái không đơn trị: không | 4 |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2. Đúng `class Solution`, đúng `countUnivalSubtrees(self, root: Optional[TreeNode]) -> int`.

In [13]:
class Solution:
    def countUnivalSubtrees(self, root: Optional[TreeNode]) -> int:
        ans = 0

        def dfs(node: Optional[TreeNode]) -> bool:
            nonlocal ans
            if node is None:
                return True
            l, r = dfs(node.left), dfs(node.right)       # xử lý cả hai nhánh trước (đếm đủ)
            if not l or not r:
                return False
            if node.left and node.left.val != node.val:
                return False
            if node.right and node.right.val != node.val:
                return False
            ans += 1
            return True

        dfs(root)
        return ans


sol = Solution()
for values, expected in examples:
    assert sol.countUnivalSubtrees(build_tree(values)) == expected
print("OK: 3 ví dụ của đề")

OK: 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Quy nạp theo chiều cao:** `dfs(node)` trả `True` ⟺ cây con tại `node` đơn trị.
  - Cơ sở: `None` → `True` (không có nút nào để vi phạm); lá: không có con nên qua mọi kiểm tra ⇒ đơn trị (đúng).
  - Bước quy nạp: nếu cây con tại `node` đơn trị thì mọi nút bằng `node.val`, nên mỗi con (nếu có) đơn trị **và** `child.val = node.val`. Ngược lại: nếu cả hai con đơn trị và `child.val == node.val`, thì mọi nút trong cây con con đều bằng `child.val = node.val` ⇒ cả cây con bằng `node.val`.
- **Đếm đủ:** mỗi nút gọi `dfs` trên **cả hai con trước khi trả về**, nên mọi nút được thăm đúng một lần và mỗi nút đơn trị cộng đúng `1`.
- **Cây rỗng không đếm:** chỉ cộng `ans` ở nhánh nút thật.
- **Cách 3, 4** thực hiện cùng quy nạp nhưng khác cách truyền kết quả (tuple / từ điển theo thứ tự hậu thứ tự).

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút, `h` là chiều cao (`h = O(log n)` cây cân bằng, `h = n` cây lệch).

| Cách | Time | Space |
|---|---|---|
| 1. Kiểm tra từng cây con | `O(n·h)` (worst `O(n²)`) | `O(h)` |
| 2. DFS bool + biến ngoài | `O(n)` | `O(h)` |
| 3. DFS trả `(cnt, uni)` | `O(n)` | `O(h)` |
| 4. Hậu thứ tự lặp | `O(n)` | `O(n)` |

Với `n <= 1000`, mọi cách đều nhanh; điểm khác biệt là **bậc**: Cách 1 tăng bình phương khi cây lệch và mọi giá trị bằng nhau (mỗi cây con phải duyệt hết). Không có khác biệt average/worst ở Cách 2–4.

## 12. Bộ test

`oracle` là brute force độc lập: với mỗi nút, **gom mọi giá trị** của cây con bằng stack tự quản rồi kiểm tra `len(set) == 1` (khác hẳn đệ quy `is_uni`). Ta đối chiếu mọi cách với `oracle` trên cây ngẫu nhiên (sinh bằng cách gắn nút ngẫu nhiên).
#### Code demo

In [14]:
def all_nodes(root):
    out, stack = [], ([root] if root else [])
    while stack:
        node = stack.pop()
        out.append(node)
        for c in (node.left, node.right):
            if c is not None:
                stack.append(c)
    return out


def oracle(root):
    """Với mỗi nút: gom mọi giá trị của cây con bằng stack tự quản, kiểm tra tập giá trị chỉ có 1 phần tử."""
    total = 0
    for node in all_nodes(root):
        values = {n.val for n in all_nodes(node)}
        if len(values) == 1:
            total += 1
    return total


def random_tree(n, alphabet, rng=random):
    """Cây nhị phân ngẫu nhiên n nút: mỗi nút mới gắn vào một chỗ trống ngẫu nhiên."""
    if n == 0:
        return None
    root = TreeNode(rng.choice(alphabet))
    slots = [(root, "left"), (root, "right")]
    for _ in range(n - 1):
        idx = rng.randrange(len(slots))
        slots[idx], slots[-1] = slots[-1], slots[idx]
        parent, side = slots.pop()
        child = TreeNode(rng.choice(alphabet))
        setattr(parent, side, child)
        slots.append((child, "left"))
        slots.append((child, "right"))
    return root


impls = {
    "bruteforce": count_bruteforce,
    "dfs_bool": count_dfs_bool,
    "pure": count_pure,
    "iterative": count_iterative,
    "submit": sol.countUnivalSubtrees,
}

for values, expected in examples:
    root = build_tree(values)
    assert oracle(root) == expected
    for key, fn in impls.items():
        assert fn(root) == expected, (key, values, fn(root), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [15]:
edge_cases = [
    ([], 0, "cây rỗng"),
    ([1], 1, "một nút"),
    ([1, 1], 2, "hai nút bằng nhau"),
    ([1, 2], 1, "hai nút khác nhau: chỉ lá 2"),
    ([1, 2, 3], 2, "gốc khác hai lá: hai lá đơn trị"),
    ([2, 2, 2], 3, "ba nút bằng nhau"),
    ([5, 5, None, 1], 1, "con bằng cha nhưng cháu khác: chỉ lá 1 đơn trị"),
    ([-1, -1, -1], 3, "giá trị âm"),
    ([1000, -1000, 1000], 2, "giá trị biên của đề"),
    ([0, 0, 0, 0, 0, 0, 0], 7, "cây hoàn hảo toàn 0"),
    ([1, 2, 2, 3, 3, 3, 3], 4, "bốn lá 3 đơn trị; hai nút 2 có hai con 3 nên không; gốc không"),
]
for values, expected, label in edge_cases:
    root = build_tree(values)
    assert oracle(root) == expected, (label, oracle(root), expected)
    for key, fn in impls.items():
        assert fn(root) == expected, (key, label, fn(root), expected)
    print("PASS", label)

PASS cây rỗng
PASS một nút
PASS hai nút bằng nhau
PASS hai nút khác nhau: chỉ lá 2
PASS gốc khác hai lá: hai lá đơn trị
PASS ba nút bằng nhau
PASS con bằng cha nhưng cháu khác: chỉ lá 1 đơn trị
PASS giá trị âm
PASS giá trị biên của đề
PASS cây hoàn hảo toàn 0
PASS bốn lá 3 đơn trị; hai nút 2 có hai con 3 nên không; gốc không


### 12.2. Tính chất và phản ví dụ

- Đáp án luôn `>=` số lá (mọi lá đơn trị) và `<= n`.
- Nếu **mọi giá trị bằng nhau** thì đáp án `= n`.
- Nếu **mọi giá trị đôi một khác nhau** thì đáp án `=` số lá.
- **Phản ví dụ** cho lỗi chỉ so giá trị cha với con: `[5,5,None,1]` (cha `5`, con `5`, cháu `1`): đáp án đúng `1`, nhưng bản chỉ so `node.val == child.val` đếm thêm.
- **Phản ví dụ** cho lỗi ngắt sớm bằng `or`: bỏ sót đếm nhánh phải.
#### Code demo

In [16]:
random.seed(250)
for _ in range(300):
    n = random.randint(1, 40)
    root = random_tree(n, [1, 2, 3])
    ans = sol.countUnivalSubtrees(root)
    leaves = sum(1 for node in all_nodes(root) if node.left is None and node.right is None)
    assert leaves <= ans <= n
    same = random_tree(n, [7])
    assert sol.countUnivalSubtrees(same) == n
print("PASS tính chất (>= số lá, <= n, toàn bằng nhau = n)")


def distinct_tree(n, rng):
    values = list(range(1, n + 1))
    rng.shuffle(values)
    it = iter(values)
    root = TreeNode(next(it))
    slots = [(root, "left"), (root, "right")]
    for _ in range(n - 1):
        parent, side = slots.pop(rng.randrange(len(slots)))
        child = TreeNode(next(it))
        setattr(parent, side, child)
        slots += [(child, "left"), (child, "right")]
    return root


rng = random.Random(2500)
for _ in range(200):
    n = rng.randint(1, 40)
    root = distinct_tree(n, rng)
    leaves = sum(1 for node in all_nodes(root) if node.left is None and node.right is None)
    assert sol.countUnivalSubtrees(root) == leaves
print("PASS giá trị đôi một khác nhau: đáp án = số lá")


def sai_chi_so_voi_con(root):
    count = 0

    def dfs(node):
        nonlocal count
        if node is None:
            return
        dfs(node.left)
        dfs(node.right)
        if (node.left is None or node.left.val == node.val) and (node.right is None or node.right.val == node.val):
            count += 1                                   # SAI: không yêu cầu con đơn trị

    dfs(root)
    return count


bad = build_tree([5, 5, None, 1])
print("phản ví dụ [5,5,null,1]: đúng", sol.countUnivalSubtrees(bad), "| chỉ so với con:", sai_chi_so_voi_con(bad))
assert sai_chi_so_voi_con(bad) != sol.countUnivalSubtrees(bad)

PASS tính chất (>= số lá, <= n, toàn bằng nhau = n)
PASS giá trị đôi một khác nhau: đáp án = số lá
phản ví dụ [5,5,null,1]: đúng 1 | chỉ so với con: 2


### 12.3. Đối chiếu ngẫu nhiên (có seed) và cây sâu
#### Code demo

In [17]:
random.seed(4250)
for trial in range(1500):
    n = random.randint(0, 25)
    alphabet = random.choice([[1], [1, 2], [1, 2, 3], list(range(-3, 4))])
    root = random_tree(n, alphabet)
    truth = oracle(root)
    for key, fn in impls.items():
        assert fn(root) == truth, (key, to_list(root), fn(root), truth)
print("PASS 1500 cây ngẫu nhiên (alphabet nhỏ, nhiều cây con đơn trị): mọi cách khớp oracle")

# Cây rất sâu (lệch trái, 900 nút ~ gần giới hạn đệ quy) và toàn bằng nhau
deep = None
for _ in range(900):
    deep = TreeNode(3, deep, None)
assert sol.countUnivalSubtrees(deep) == 900 == count_iterative(deep) == count_pure(deep)
print("PASS cây lệch sâu 900 nút toàn bằng nhau: đáp án 900")

PASS 1500 cây ngẫu nhiên (alphabet nhỏ, nhiều cây con đơn trị): mọi cách khớp oracle
PASS cây lệch sâu 900 nút toàn bằng nhau: đáp án 900


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ so giá trị cha với con mà không yêu cầu con đơn trị
Xem `sai_chi_so_voi_con` ở phần test: `[5,5,None,1]` bị đếm nhầm gốc `5` (con `5` bằng cha) dù cháu `1` khác. **Code đúng:** thêm điều kiện `con đơn trị` (kết quả của `dfs(con)`).

### Lỗi 2: Ngắt sớm bằng `or` (bỏ sót đếm nhánh phải)
#### Code sai

In [18]:
def sai_ngat_som(root):
    count = 0

    def dfs(node):
        nonlocal count
        if node is None:
            return True
        if not dfs(node.left) or not dfs(node.right):         # SAI: nếu trái False thì phải KHÔNG được gọi
            return False
        if node.left and node.left.val != node.val:
            return False
        if node.right and node.right.val != node.val:
            return False
        count += 1
        return True

    dfs(root)
    return count


t = build_tree([1, 2, 3])        # lá 2 và 3 đơn trị; gốc 1 không
t2 = build_tree([1, 1, 2, 2, 2, 1, 1])      # nhánh trái sai ở dưới, nhánh phải còn nhiều cây con đơn trị
print("đúng:", sol.countUnivalSubtrees(t), "| sai:", sai_ngat_som(t))
print("đúng:", sol.countUnivalSubtrees(t2), "| sai:", sai_ngat_som(t2))
assert sai_ngat_som(t2) != sol.countUnivalSubtrees(t2)

đúng: 2 | sai: 2
đúng: 4 | sai: 2


**Vì sao sai:** biểu thức `not dfs(trái) or not dfs(phải)` **không gọi** `dfs(phải)` khi `dfs(trái)` là `False` (đoản mạch). Các nút ở nhánh phải không được thăm nên không được đếm. **Code đúng:** `l, r = dfs(trái), dfs(phải)` rồi mới kiểm tra.

### Lỗi 3: So `None` với giá trị (hoặc gọi `.val` của `None`)
#### Code sai

In [19]:
def sai_none(root):
    count = 0

    def dfs(node):
        nonlocal count
        if node is None:
            return True
        l, r = dfs(node.left), dfs(node.right)
        if not l or not r:
            return False
        if node.left.val != node.val:                         # SAI: node.left có thể là None
            return False
        count += 1
        return True

    dfs(root)
    return count


try:
    sai_none(build_tree([1, None, 1]))
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'NoneType' object has no attribute 'val'


**Vì sao sai:** lá hoặc nút thiếu con có `left`/`right` là `None`; `None.val` gây `AttributeError`. **Code đúng:** `if node.left and node.left.val != node.val` (hoặc `node.left is not None`).

### Lỗi 4: Chỉ đếm lá (hoặc đếm cả `None`)
#### Code sai

In [20]:
def sai_dem_la(root):
    return sum(1 for node in all_nodes(root) if node.left is None and node.right is None)


def sai_dem_none(root):
    def dfs(node):
        if node is None:
            return 1                                          # SAI: cây rỗng không phải một cây con
        return dfs(node.left) + dfs(node.right) + 1
    return dfs(root)


r = build_tree([5, 5, 5, 5, 5, None, 5])
print("đúng:", sol.countUnivalSubtrees(r), "| chỉ lá:", sai_dem_la(r))
print("số nút (n):", len(all_nodes(r)), "| đếm cả None:", sai_dem_none(r))

đúng: 6 | chỉ lá: 3
số nút (n): 6 | đếm cả None: 13


**Vì sao sai:** các nút **trong** (có con) cũng có thể đơn trị (ví dụ ví dụ 3 có `6`, không chỉ `3` lá); và `None` không phải một cây con của đề (cây rỗng). Chỉ đếm khi gặp **nút thật** đơn trị.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**đếm / kiểm tra mọi cây con** thỏa tính chất P" mà P của cha **phụ thuộc P của con** → **DFS hậu thứ tự** (con trả lời trước).
- Hàm đệ quy trả về **cả kết quả đếm lẫn cờ trạng thái** (tuple) thay vì dùng biến ngoài khi muốn code thuần.
- **Không dùng `or` / `and` đoản mạch** khi cả hai vế đều có tác dụng phụ (đếm, gán): hãy gọi riêng rẽ rồi mới kết hợp.
- Chuỗi "con đơn trị **và** giá trị con bằng cha" gói gọn bài; tương tự: **Balanced Binary Tree** (con cân bằng **và** chênh lệch chiều cao ≤ 1), **Validate BST** (khoảng giá trị).
- Cây sâu (≥ 1000) → coi chừng **giới hạn đệ quy** của Python; dùng bản lặp (Cách 4) nếu cần.
- Luôn thử `[]`, một nút, "con bằng cha nhưng cháu khác", toàn bằng nhau, toàn khác nhau.

## 15. Biến thể của bài

1. **Largest Univalue Path** (LeetCode 687): đường đi dài nhất cùng giá trị.
2. **Balanced Binary Tree** (110), **Diameter of Binary Tree** (543): cùng khung hậu thứ tự trả về nhiều giá trị.
3. **Count Good Nodes in Binary Tree** (1448): đếm theo thông tin truyền xuống (tiền thứ tự).
4. **Univalue Binary Tree** (965): kiểm tra cả cây có đơn trị.
5. Đếm cây con **gồm toàn số chẵn / toàn dấu như nhau**; đếm cây con **kích thước lớn nhất đơn trị**.
#### Code demo

In [21]:
def largest_univalue_subtree_size(root):
    """Kích thước lớn nhất của một cây con đơn trị (hậu thứ tự trả về (size, uni))."""
    best = 0

    def f(node):
        nonlocal best
        if node is None:
            return 0, True
        sl, ul = f(node.left)
        sr, ur = f(node.right)
        uni = ul and ur and (node.left is None or node.left.val == node.val) and (node.right is None or node.right.val == node.val)
        size = sl + sr + 1
        if uni:
            best = max(best, size)
        return size, uni

    f(root)
    return best


def is_univalue_tree(root):
    return root is None or all(n.val == root.val for n in all_nodes(root))


print(largest_univalue_subtree_size(build_tree([5, 1, 5, 5, 5, None, 5])))    # 2 (nút 5 phải và con của nó)
print(largest_univalue_subtree_size(build_tree([5, 5, 5, 5, 5, None, 5])))    # 6
print(is_univalue_tree(build_tree([2, 2, 2])), is_univalue_tree(build_tree([2, 2, 3])))

2
6
True False


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Kiểm tra từng cây con | với mỗi nút duyệt hết cây con | `O(n·h)` | `O(h)` | oracle, dễ hiểu |
| 2. DFS bool + biến ngoài | con trả cờ đơn trị, cha so giá trị | `O(n)` | `O(h)` | **cách nộp chuẩn** |
| 3. DFS trả `(cnt, uni)` | không biến ngoài | `O(n)` | `O(h)` | code thuần, dễ test |
| 4. Hậu thứ tự lặp | stack + từ điển kết quả | `O(n)` | `O(n)` | cây rất sâu, tránh đệ quy |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `to_list(build_tree([1, None, 2]))`; (b) `Solution().countUnivalSubtrees(build_tree([1]))`; (c) `Solution().countUnivalSubtrees(build_tree([1, 1, 1]))`; (d) `Solution().countUnivalSubtrees(build_tree([1, 2, 1]))`.

**Bài 2 (tính bằng tay).** Vẽ cây `[2,2,2,2,3,2,2]` và liệt kê các nút có cây con đơn trị.

**Bài 3 (điền TODO).** Hoàn thiện `count_todo(root)` theo Cách 3 (trả `(cnt, uni)`).
#### Code demo

In [22]:
def count_todo(root):
    def f(node):
        # TODO 1: nếu node là None -> trả (0, True)
        # TODO 2: (cl, ul) = f(node.left); (cr, ur) = f(node.right)
        # TODO 3: uni = ul and ur and (node.left là None hoặc bằng node.val) and (node.right là None hoặc bằng node.val)
        # TODO 4: trả (cl + cr + (1 nếu uni), uni)
        pass

    # TODO 5: trả f(root)[0]
    pass


# Tự làm xong rồi mới mở test:
# for values, expected in examples:
#     assert count_todo(build_tree(values)) == expected

**Bài 4 (tự viết lại).** Viết `count_iter2(root)` bằng hậu thứ tự lặp, **không nhìn lại** Cách 4; kiểm tra bằng `oracle` trên cây ngẫu nhiên.

**Bài 5 (biến thể / mini challenge).** Viết `count_univalue_paths_rooted(root)`: đếm số **nút** `v` mà **mọi đường từ `v` xuống lá** đều có cùng giá trị — đó chính là cây con đơn trị; tiếp theo viết `count_non_univalue(root)` đếm số cây con **không** đơn trị và kiểm tra `count_univalue + count_non_univalue == n`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `build_tree` điền con trái trước, `None` là rỗng; lá luôn đơn trị. Bài 2: tìm các lá trước, rồi lên cha. Bài 5: số cây con = số nút `n`.
- **Mức 2 (vừa):** Bài 1: `[1, None, 2]`, `1`, `3`, `2`. Bài 2: cây có bốn lá (`2`, `3`, `2`, `2`) đều đơn trị; ở tầng giữa, nút trái `2` có con `2` và `3` ⇒ không; nút phải `2` có con `2`, `2` ⇒ có. Bài 3: khung Cách 3.
- **Mức 3 (sâu):** Bài 2: các nút đơn trị: `4` lá + nút `2` bên phải = `5`; nút `2` bên trái (con `2`, `3`) và gốc (con trái không đơn trị) thì không. Bài 5: `count_non_univalue = n - count_univalue`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [23]:
# Bài 1
assert to_list(build_tree([1, None, 2])) == [1, None, 2]
assert Solution().countUnivalSubtrees(build_tree([1])) == 1
assert Solution().countUnivalSubtrees(build_tree([1, 1, 1])) == 3
assert Solution().countUnivalSubtrees(build_tree([1, 2, 1])) == 2

# Bài 2
t2 = build_tree([2, 2, 2, 2, 3, 2, 2])
assert oracle(t2) == 5 == Solution().countUnivalSubtrees(t2)


# Bài 3
def count_todo_ans(root):
    def f(node):
        if node is None:
            return 0, True
        cl, ul = f(node.left)
        cr, ur = f(node.right)
        uni = ul and ur and (node.left is None or node.left.val == node.val) and (node.right is None or node.right.val == node.val)
        return cl + cr + (1 if uni else 0), uni

    return f(root)[0]


for values, expected in examples:
    assert count_todo_ans(build_tree(values)) == expected


# Bài 4
def count_iter2(root):
    if root is None:
        return 0
    order, stack = [], [root]
    while stack:
        node = stack.pop()
        order.append(node)
        stack += [c for c in (node.left, node.right) if c]
    good, total = {}, 0
    for node in reversed(order):
        good[node] = all(c is None or (good[c] and c.val == node.val) for c in (node.left, node.right))
        total += good[node]
    return total


random.seed(9)
for _ in range(300):
    t = random_tree(random.randint(0, 25), [1, 2])
    assert count_iter2(t) == oracle(t)


# Bài 5
def count_non_univalue(root):
    return len(all_nodes(root)) - Solution().countUnivalSubtrees(root)


for _ in range(100):
    t = random_tree(random.randint(0, 25), [1, 2])
    assert Solution().countUnivalSubtrees(t) + count_non_univalue(t) == len(all_nodes(t))
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hậu thứ tự: con trả lời trước)

```python
class Solution:
    def countUnivalSubtrees(self, root: Optional[TreeNode]) -> int:
        ans = 0

        def dfs(node):
            nonlocal ans
            if node is None:
                return True
            l, r = dfs(node.left), dfs(node.right)      # gọi cả hai, không đoản mạch
            if not l or not r:
                return False
            if node.left and node.left.val != node.val:
                return False
            if node.right and node.right.val != node.val:
                return False
            ans += 1
            return True

        dfs(root)
        return ans
```

**Checklist**

- [ ] Cây con tại `v` gồm `v` **và mọi con cháu**; mỗi nút một cây con; `None` không tính?
- [ ] Đơn trị ⟺ **hai con đơn trị** **và** giá trị con (nếu có) `== node.val`?
- [ ] Gọi `dfs` cho **cả hai** con trước khi quyết định (không đoản mạch bằng `or`)?
- [ ] Không truy cập `.val` của `None`?
- [ ] Đã thử `[]`, một nút, `[5,5,None,1]`, toàn bằng nhau, toàn khác nhau?
- [ ] Đã đối chiếu với brute force gom giá trị từng cây con trên cây ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def countUnivalSubtrees(self, root: Optional[TreeNode]) -> int`?